# Week 1: Delivery-location clustering for logistics planning

## Objective and data
Explore whether validated delivery destinations form geographic patterns that may help structure later service-area planning. The input is `data/cleaned_logistics_data.csv`. Only records with valid restaurant and delivery coordinate flags and nonzero coordinates within broad India bounds are used. Restaurant coordinates are used as a validation gate; clustering is based primarily on delivery-location latitude and longitude.

## Method: DBSCAN with geographic distance
DBSCAN is a density-based method that can identify irregularly shaped groups and label isolated records as noise. Configuration: **eps = 1.0 km**, the maximum great-circle neighborhood radius; **min_samples = 20**, the number of nearby deliveries (including the point itself) needed to start a dense cluster. One kilometre is a local geographic neighborhood scale for exploratory service-area analysis, and 20 records reduces the chance of declaring very sparse points to be core areas. This is a reasoned starting configuration, not a universally optimal setting; results depend on these values.

Latitude/longitude are not treated as ordinary Cartesian coordinates. When available, the script uses scikit-learn DBSCAN's Haversine metric. In this environment it instead uses a SciPy spatial tree over 3-D unit-sphere coordinates: for angular Haversine radius ε, chord radius `2 sin(ε/2)` gives the equivalent spherical neighbor condition.

Clusters are exploratory geographic patterns, **not official dispatch zones**. A production zoning decision would need road-network information, actual travel times, demand by time, driver availability, fleet capacity, and operational constraints.

In [1]:
import sys
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/'src/clustering.py').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from clustering import run_clustering
results=run_clustering(ROOT/'data/cleaned_logistics_data.csv',eps_km=1.0,min_samples=20,make_figures=True)
summary=results['summary']
print('Implementation:',results['implementation'])
print(f"Valid geographic records: {results['valid_records']:,}")
print(f"Configuration: eps={results['eps_km']:.1f} km; min_samples={results['min_samples']}")
print(f"Number of clusters: {results['cluster_count']}")
print(f"Noise/outliers: {results['noise_count']:,} ({results['noise_percent']:.2f}%)")
print('Cluster details: size; average and median delivery time; average approximate straight-line distance; median destination coordinates')
display(summary)
sizes=summary[summary.records>=100]
print(f"Clusters with at least 100 records: {len(sizes)} of {len(summary)}")
print('Largest meaningful clusters by mean delivery time (descriptive only):')
display(sizes.sort_values('average_delivery_time_min',ascending=False).head(5))
print('Largest meaningful clusters by lowest mean delivery time (descriptive only):')
display(sizes.sort_values('average_delivery_time_min').head(5))

Implementation: SciPy cKDTree DBSCAN (Haversine-equivalent spherical chord distance)
Valid geographic records: 41,513
Configuration: eps=1.0 km; min_samples=20
Number of clusters: 280
Noise/outliers: 7,708 (18.57%)
Cluster details: size; average and median delivery time; average approximate straight-line distance; median destination coordinates
         records  average_delivery_time_min  median_delivery_time_min  average_approx_distance_km  median_latitude  median_longitude  share_of_valid_records_pct
cluster                                                                                                                                                         
13          2665                      25.41                      25.0                        8.00            21.22             72.84                        6.42
35          2457                      25.63                      25.0                        8.01            22.79             75.94                        5.92
21       

## Geographic pattern
The map colors each validated destination by DBSCAN cluster. Gray points are noise under the chosen density rule. Dense groups suggest repeated destination areas; many labels or gray points show where this single-radius configuration does not form a sufficiently dense group. Coordinates are plotted geographically for visualization; DBSCAN itself used spherical distance.

![Delivery locations colored by DBSCAN cluster](../visualizations/delivery_location_dbscan_clusters.png)

## Cluster sizes and service performance
The size and delivery-time charts show the 20 largest clusters for readability. The notebook output above includes all discovered clusters and their record counts, mean delivery time, and average approximate distance. Compare groups with adequate sample sizes; differences are descriptive associations and do not mean location caused a delivery-time difference.

![Cluster size comparison](../visualizations/delivery_cluster_sizes.png)

![Average delivery time by cluster](../visualizations/delivery_cluster_average_time.png)

![Average approximate distance by cluster](../visualizations/delivery_cluster_average_distance.png)

## Logistics strategy: how to use these patterns
- **Service-area planning:** treat dense destination neighborhoods as candidate areas for further review, then redraw boundaries using road travel times and service policies.
- **Driver allocation:** combine zone-time demand with courier shifts, vehicle type, and capacity to plan staffing. The present records do not include live availability or capacity.
- **Demand monitoring:** track order counts per geographic area and time period to identify growth or peaks; this notebook clusters all available observations together and does not model time-varying demand.
- **Delivery-time analysis:** compare service-time summaries across sufficiently populated clusters to target operational investigation. Cluster membership is not a cause of longer/shorter time.
- **Future route optimization:** use cleaned service areas as a possible input after obtaining road-network travel times, batch/sequence details, fleet constraints, and operational rules. No optimization is performed here.

Noise points may be valid but isolated deliveries, low-density locations, or coordinate anomalies that survived broad validation. They are not automatically errors. Straight-line delivery distance in the summaries is approximate, not road mileage.

## Limitations
DBSCAN's cluster count and noise share depend strongly on `eps` and `min_samples`; this is one transparent exploratory configuration, not an optimized choice. Latitude/longitude validity relies on the preprocessing flags and broad geographic bounds. The dataset lacks road-network information, actual travel times, demand timestamps at operational granularity, driver availability, fleet capacity, and dispatch constraints. Therefore, these clusters are not official operating territories and should not be used directly for dispatch.